# LOAD DATASET

In [4]:
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [9]:
X_train = pd.read_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/X_train.csv')
X_test = pd.read_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/X_test.csv')

y_train = pd.read_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/y_train.csv')
y_test = pd.read_csv('C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Datasets/y_test.csv')

In [10]:
display(X_train.info())
display(X_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63016 entries, 0 to 63015
Data columns (total 82 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   gender                       63016 non-null  int64  
 1   is_home                      63016 non-null  int64  
 2   neutral                      63016 non-null  int64  
 3   tournament                   63016 non-null  float64
 4   venue_country                63016 non-null  float64
 5   days_since_last_match_team   63016 non-null  float64
 6   days_since_last_match_opp    63016 non-null  float64
 7   elo_team                     63016 non-null  float64
 8   elo_opponent                 63016 non-null  float64
 9   population_team              63016 non-null  float64
 10  population_opp               63016 non-null  float64
 11  gdp_per_capita_team          63016 non-null  float64
 12  gdp_per_capita_opp           63016 non-null  float64
 13  altitude_venue  

None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15756 entries, 0 to 15755
Data columns (total 82 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   gender                       15756 non-null  int64  
 1   is_home                      15756 non-null  int64  
 2   neutral                      15756 non-null  int64  
 3   tournament                   15756 non-null  float64
 4   venue_country                15756 non-null  float64
 5   days_since_last_match_team   15756 non-null  float64
 6   days_since_last_match_opp    15756 non-null  float64
 7   elo_team                     15756 non-null  float64
 8   elo_opponent                 15756 non-null  float64
 9   population_team              15756 non-null  float64
 10  population_opp               15756 non-null  float64
 11  gdp_per_capita_team          15756 non-null  float64
 12  gdp_per_capita_opp           15756 non-null  float64
 13  altitude_venue  

None

In [11]:
display(y_train.info())
display(y_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63016 entries, 0 to 63015
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   team_goals  63016 non-null  float64
 1   opp_goals   63016 non-null  float64
dtypes: float64(2)
memory usage: 984.8 KB


None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15756 entries, 0 to 15755
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   team_goals  15756 non-null  float64
 1   opp_goals   15756 non-null  float64
dtypes: float64(2)
memory usage: 246.3 KB


None

# BASELINE HYPERPARAMETER

In [1]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [6]:
rf = RandomForestRegressor(random_state=42)
xgb = XGBRegressor(random_state=42)
lgbm = LGBMRegressor(random_state=42)
gb = GradientBoostingRegressor(random_state=42)
ab = AdaBoostRegressor(random_state=42)

In [7]:
from sklearn.multioutput import RegressorChain

chain_rf = RegressorChain(base_estimator=rf, order=[0, 1])
chain_xgb = RegressorChain(base_estimator=xgb, order=[0, 1])
chain_lgbm = RegressorChain(base_estimator=lgbm, order=[0, 1])
chain_gb = RegressorChain(base_estimator=gb, order=[0, 1])
chain_ab = RegressorChain(base_estimator=ab, order=[0, 1])

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    'Random Forest': chain_rf,
    'XGBoost': chain_xgb,
    'LightGBM': chain_lgbm,
    'Gradient Boosting': chain_gb,
    'Ada Boost': chain_ab
}

In [12]:
from sklearn.model_selection import KFold, cross_validate

kf = KFold(n_splits=3, shuffle=True, random_state=42)

result = []

for name, model in models.items():
    cv_scores = cross_validate(model,
                               X_train, y_train,
                               cv=kf,
                               scoring=['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
                               n_jobs=-1)
    result.append({
        'Model': name,
        'RMSE': -cv_scores['test_neg_root_mean_squared_error'].mean(),
        'MAE': -cv_scores['test_neg_mean_absolute_error'].mean(),
        'R2': cv_scores['test_r2'].mean()
    })

df_results = pd.DataFrame(result).sort_values(by='R2', ascending=False).reset_index(drop=True)

df_results

,Model,RMSE,MAE,R2
0,LightGBM,1.465636,1.037076,0.327486
1,Gradient Boosting,1.490493,1.050116,0.304472
2,XGBoost,1.497162,1.056711,0.298229
3,Random Forest,1.499620,1.065395,0.295907
4,Ada Boost,2.579901,2.264074,-1.116485


In [13]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

result_2 = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_round = np.round(y_pred).astype(int)

    rmse = root_mean_squared_error(y_test, y_pred_round)
    mae = mean_absolute_error(y_test, y_pred_round)
    r2 = r2_score(y_test, y_pred_round)

    result_2.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    })

df_results_2 = pd.DataFrame(result_2).sort_values('R2', ascending=False).reset_index(drop=True)
df_results_2

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014236 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8927
[LightGBM] [Info] Number of data points in the train set: 63016, number of used features: 81
[LightGBM] [Info] Start training from score 1.560715
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005285 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8951
[LightGBM] [Info] Number of data points in the train set: 63016, number of used features: 82
[LightGBM] [Info] Start training from score 1.560715


c:\Users\jojo0\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jojo0\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jojo0\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jojo0\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Model,RMSE,MAE,R2
0,LightGBM,1.520174,1.029386,0.295202
1,Gradient Boosting,1.539480,1.037510,0.277162
2,Random Forest,1.543043,1.054582,0.273813
3,XGBoost,1.550429,1.045792,0.266855
4,Ada Boost,2.763086,2.409590,-1.344012


# HYPERPARAMETER TUNING

In [15]:
from sklearn.model_selection import GridSearchCV

## LightGBM

In [19]:
lgbm_params_1 = {
    "base_estimator__num_leaves": [20, 30, 50, 80],
    "base_estimator__max_depth": [3, 5, 8, 10, 12, 15]
}

grid_lgbm_1 = GridSearchCV(
    estimator=chain_lgbm,
    param_grid=lgbm_params_1,
    scoring='r2',
    cv=kf,
    n_jobs=-1,
    verbose=2
)

grid_lgbm_1.fit(X_train, y_train)

best_params_lgbm = grid_lgbm_1.best_params_

display(grid_lgbm_1.best_params_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014372 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8927
[LightGBM] [Info] Number of data points in the train set: 63016, number of used features: 81
[LightGBM] [Info] Start training from score 1.560715
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014456 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8951
[LightGBM] [Info] Number of data points in the train set: 63016, number of used features: 82
[LightGBM] [Info] Start training from score 1.560715
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


{'base_estimator__max_depth': 10, 'base_estimator__num_leaves': 50}

In [29]:
lgbm_params_2 = {
    "base_estimator__subsample": [0.4, 0.6, 0.8, 1.0],
    "base_estimator__colsample_bytree": [0.4, 0.6, 0.8, 1.0]
}

lgbm_2 = LGBMRegressor(random_state=42)
chain_lgbm_2 = RegressorChain(base_estimator=lgbm_2, order=[0, 1])
chain_lgbm_2.set_params(**best_params_lgbm)

grid_lgbm_2 = GridSearchCV(
    estimator=chain_lgbm_2,
    param_grid=lgbm_params_2,
    scoring='r2',
    cv=kf,
    n_jobs=-1,
    verbose=2
)

grid_lgbm_2.fit(X_train, y_train)

best_params_lgbm.update(grid_lgbm_2.best_params_)

display(grid_lgbm_2.best_params_)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003295 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8927
[LightGBM] [Info] Number of data points in the train set: 63016, number of used features: 81
[LightGBM] [Info] Start training from score 1.560715
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGB

{'base_estimator__colsample_bytree': 0.4, 'base_estimator__subsample': 0.4}

In [30]:
lgbm_params_3 = {
    "base_estimator__min_child_samples": [10, 20, 50, 100, 300, 500],
    "base_estimator__reg_lambda": [0.01, 0.1, 0.5, 1, 5, 10]
}

lgbm_3 = LGBMRegressor(random_state=42)
chain_lgbm_3 = RegressorChain(base_estimator=lgbm_3, order=[0, 1])
chain_lgbm_3.set_params(**best_params_lgbm)

grid_lgbm_3 = GridSearchCV(
    estimator=chain_lgbm_3,
    param_grid=lgbm_params_3,
    scoring='r2',
    cv=kf,
    n_jobs=-1,
    verbose=2
)

grid_lgbm_3.fit(X_train, y_train)

best_params_lgbm.update(grid_lgbm_3.best_params_)

display(grid_lgbm_3.best_params_)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003216 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8927
[LightGBM] [Info] Number of data points in the train set: 63016, number of used features: 81
[LightGBM] [Info] Start training from score 1.560715
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

{'base_estimator__min_child_samples': 10, 'base_estimator__reg_lambda': 10}

In [31]:
lgbm_params_4 = [
    {"base_estimator__learning_rate": [0.1], "base_estimator__n_estimators": [100, 200]},
    {"base_estimator__learning_rate": [0.05], "base_estimator__n_estimators": [300, 500]},
    {"base_estimator__learning_rate": [0.01], "base_estimator__n_estimators": [800, 1000]}
]

lgbm_4 = LGBMRegressor( random_state=42)
chain_lgbm_4 = RegressorChain(base_estimator=lgbm_4, order=[0, 1])
chain_lgbm_4.set_params(**best_params_lgbm)

grid_lgbm_4 = GridSearchCV(
    estimator=chain_lgbm_4,
    param_grid=lgbm_params_4,
    scoring='r2',
    cv=kf,
    n_jobs=-1,
    verbose=2
)

grid_lgbm_4.fit(X_train, y_train)

best_params_lgbm.update(grid_lgbm_4.best_params_)

display(grid_lgbm_4.best_params_)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002699 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8927
[LightGBM] [Info] Number of data points in the train set: 63016, number of used features: 81
[LightGBM] [Info] Start training from score 1.560715
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

{'base_estimator__learning_rate': 0.05, 'base_estimator__n_estimators': 500}

In [32]:
lgbm_tuned = LGBMRegressor(random_state=42)
chain_lgbm_tuned = RegressorChain(base_estimator=lgbm_tuned, order=[0, 1])
chain_lgbm_tuned.set_params(**best_params_lgbm)

chain_lgbm_tuned.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012027 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8927
[LightGBM] [Info] Number of data points in the train set: 63016, number of used features: 81
[LightGBM] [Info] Start training from score 1.560715
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

RegressorChain(base_estimator=LGBMRegressor(colsample_bytree=0.4,
                                            learning_rate=0.05, max_depth=10,
                                            min_child_samples=10,
                                            n_estimators=500, num_leaves=50,
                                            random_state=42, reg_lambda=10,
                                            subsample=0.4),
               order=[0, 1])

## Gradient Boosting

In [27]:
gb_params_1 = {
    "base_estimator__max_depth": [3, 5, 8, 10],
    "base_estimator__min_samples_split":[3, 5, 8, 10, 12],
    "base_estimator__min_samples_leaf": [3, 5, 8, 10, 12]
}

grid_gb_1 = GridSearchCV(
    estimator=chain_gb,
    param_grid=gb_params_1,
    scoring='r2',
    cv=kf,
    n_jobs=-1,
    verbose=2
)

grid_gb_1.fit(X_train, y_train)

best_params_gb = grid_gb_1.best_params_

display(grid_gb_1.best_params_)

Fitting 3 folds for each of 100 candidates, totalling 300 fits


{'base_estimator__max_depth': 8,
 'base_estimator__min_samples_leaf': 12,
 'base_estimator__min_samples_split': 3}

In [33]:
gb_params_2 = [
    {"base_estimator__learning_rate": [0.1], "base_estimator__n_estimators": [100, 200]},
    {"base_estimator__learning_rate": [0.05], "base_estimator__n_estimators": [300, 500]},
    {"base_estimator__learning_rate": [0.01], "base_estimator__n_estimators": [800, 1000]}
]

gb_2 = GradientBoostingRegressor(random_state=42)
chain_gb_2 = RegressorChain(base_estimator=gb_2, order=[0, 1])
chain_gb_2.set_params(**best_params_gb)

grid_gb_2 = GridSearchCV(
    estimator=chain_gb_2,
    param_grid=gb_params_2,
    scoring='r2',
    cv=kf,
    n_jobs=-1,
    verbose=2
)

grid_gb_2.fit(X_train, y_train)

best_params_gb.update(grid_gb_2.best_params_)

display(grid_gb_2.best_params_)

Fitting 3 folds for each of 6 candidates, totalling 18 fits


{'base_estimator__learning_rate': 0.01, 'base_estimator__n_estimators': 1000}

In [34]:
gb_params_3 = {
    "base_estimator__subsample": [0.4, 0.6, 0.8, 1.0],
    "base_estimator__max_features": ['sqrt', 'log2', None]
}

gb_3 = GradientBoostingRegressor(random_state=42)
chain_gb_3 = RegressorChain(base_estimator=gb_3, order=[0, 1])
chain_gb_3.set_params(**best_params_gb)

grid_gb_3 = GridSearchCV(
    estimator=chain_gb_3,
    param_grid=gb_params_3,
    scoring='r2',
    cv=kf,
    n_jobs=-1,
    verbose=2
)

grid_gb_3.fit(X_train, y_train)

best_params_gb.update(grid_gb_3.best_params_)

display(grid_gb_3.best_params_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


{'base_estimator__max_features': 'sqrt', 'base_estimator__subsample': 1.0}

In [35]:
gb_tuned = GradientBoostingRegressor(random_state=42)
chain_gb_tuned = RegressorChain(base_estimator=gb_tuned, order=[0, 1])
chain_gb_tuned.set_params(**best_params_gb)

chain_gb_tuned.fit(X_train, y_train)

RegressorChain(base_estimator=GradientBoostingRegressor(learning_rate=0.01,
                                                        max_depth=8,
                                                        max_features='sqrt',
                                                        min_samples_leaf=12,
                                                        min_samples_split=3,
                                                        n_estimators=1000,
                                                        random_state=42),
               order=[0, 1])

# FINAL EVALUATION

In [36]:
tuned_models = {
    'LightGBM Regressor': chain_lgbm_tuned,
    'Gradient Boosting Regressor': chain_gb_tuned,
}

tuned_result = []

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_pred_round = np.round(y_pred).astype(int)

    rmse = root_mean_squared_error(y_test, y_pred_round)
    mae = mean_absolute_error(y_test, y_pred_round)
    r2 = r2_score(y_test, y_pred_round)

    tuned_result.append({
        'Model': name,
        'Final Test RMSE': rmse,
        'Final Test MAE': mae,
        'Final Test R2-Score': r2
    })

df_tuned_result = pd.DataFrame(tuned_result).sort_values(by='Final Test R2-Score', ascending=False).reset_index(drop=True)

df_tuned_result

c:\Users\jojo0\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jojo0\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jojo0\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jojo0\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Model,Final Test RMSE,Final Test MAE,Final Test R2-Score
0,Gradient Boosting Regressor,1.505475,1.017073,0.308752
1,LightGBM Regressor,1.507733,1.021674,0.306691


# SAVE MODEL

In [37]:
import joblib

joblib.dump(chain_lgbm_tuned, "C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Model/lgbm_model.pkl")
joblib.dump(chain_gb_tuned, "C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Model/gb_model.pkl")

['C:/Users/jojo0/OneDrive/Documents/Competition/Gammafest-2026/Model/gb_model.pkl']